# Grad-CAM: ¿en qué se fija una red neuronal para decidir? 🔍

En el workshop entrenamos una **ResNet-18** que distingue **piedra 🪨, papel 📄 y tijeras ✂️** con más de 95% de accuracy. Pero... ¿*por qué* acierta? ¿Mira la mano, o está haciendo trampa mirando otra cosa (el fondo, la iluminación, una sombra)?

Una red neuronal es una **caja negra**: le entra una imagen y le sale una respuesta, sin explicaciones. Hoy vamos a abrir esa caja con **Grad-CAM** (*Gradient-weighted Class Activation Mapping*, Selvaraju et al., 2017).

### ¿Qué produce Grad-CAM?

Un **mapa de calor** encima de la imagen que responde a la pregunta:

> *"¿Qué zonas de la imagen hicieron que la red dijera **tijeras**?"*

- 🔴 **Rojo** → zonas que empujaron fuerte hacia esa decisión.
- 🔵 **Azul** → zonas que no importaron.

### Plan del notebook

| # | Paso | ¿Qué haremos? |
|---|------|---------------|
| 1 | Preparar | Datos + modelo ResNet-18 entrenado (reusamos el workshop) |
| 2 | Entender | ¿Qué son los *feature maps* de una CNN? |
| 3 | Construir | Grad-CAM **paso a paso**, a mano, con 10 líneas de PyTorch |
| 4 | Aplicar | Galería de mapas de calor en imágenes de test |
| 5 | Investigar | ¿Por qué *tijeras* y no *piedra*? ¿Por qué se equivoca? |
| 6 | Validar | Prueba de cordura: ¿un modelo sin entrenar da mapas con sentido? |

> 💡 **Stack:** PyTorch · torchvision · timm · NumPy · matplotlib

## 1. Setup: librerías, semilla y dispositivo

Lo mismo que en el workshop: importamos librerías, fijamos la semilla (para que todos obtengamos resultados parecidos) y elegimos el dispositivo (GPU si hay, si no CPU).

La única librería nueva es `torch.nn.functional` (abreviada `F`), que usaremos para **agrandar** el mapa de calor al tamaño de la imagen.

In [ ]:
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from PIL import Image
import timm

# Semilla para que los resultados sean reproducibles
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():      # GPU de Apple (Mac M1/M2/M3...)
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():            # GPU NVIDIA
    DEVICE = torch.device("cuda")
else:                                      # CPU
    DEVICE = torch.device("cpu")

print(f"PyTorch {torch.__version__} | Dispositivo: {DEVICE}")

## 2. Los datos: piedra, papel o tijeras

Usamos el mismo dataset del workshop: ~2,200 fotos de manos sobre un **fondo verde**, organizadas en una carpeta por clase.

```
data/rock-paper-scissors/
├── paper/
├── rock/
└── scissors/
```

> **Guarda esta pregunta para después:** todas las fotos tienen el mismo fondo verde. Si la red aprendiera a mirar el fondo, no le serviría para distinguir clases... pero ¿y si hay otra "pista fácil" escondida? Grad-CAM nos lo dirá.

In [ ]:
DATA_DIR = Path("../data/rock-paper-scissors")

for carpeta in sorted(DATA_DIR.iterdir()):
    if carpeta.is_dir():
        n_imagenes = len(list(carpeta.glob("*.png")))
        print(f"{carpeta.name:>10s} → {n_imagenes} imágenes")

### Preprocesamiento (igual que en el transfer learning del workshop)

ResNet-18 fue pre-entrenada con **ImageNet**, así que le damos las imágenes como las espera:

- `Resize(224×224)` → el tamaño con el que fue entrenada.
- `ToTensor()` → imagen a tensor con valores entre 0 y 1.
- `Normalize(...)` → con la media y desviación estándar de ImageNet.

También creamos `desnormalizar()`: la operación **inversa**, para poder *mostrar* un tensor normalizado como una imagen normal. La necesitaremos para dibujar el mapa de calor encima.

In [ ]:
IMG_SIZE = 224
MEDIA_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEDIA_IMAGENET, std=STD_IMAGENET),
])


def desnormalizar(tensor):
    # Tensor normalizado [3, 224, 224] → imagen [224, 224, 3] con valores en [0, 1]
    media = torch.tensor(MEDIA_IMAGENET).view(3, 1, 1)
    std = torch.tensor(STD_IMAGENET).view(3, 1, 1)
    imagen = tensor.detach().cpu() * std + media
    return imagen.clamp(0, 1).permute(1, 2, 0).numpy()

### Dataset y DataLoaders (función reutilizada del workshop)

Reusamos `crear_dataloaders()` tal cual la dejamos al final del workshop: divide los datos en **70% train / 15% val / 15% test** con la **misma semilla**, y además devuelve las **rutas de las imágenes de test**.

¿Por qué importa usar la misma semilla? Porque así el conjunto de test es **exactamente el mismo** que en el workshop: si cargamos el modelo ya entrenado, lo estaremos evaluando con imágenes que **nunca vio**.

In [ ]:
BATCH_SIZE = 32


def crear_dataloaders(transform, batch_size=BATCH_SIZE):
    # Crea los DataLoaders de train/val/test y devuelve también las rutas de test.
    dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

    n_total = len(dataset)
    n_train = int(0.70 * n_total)
    n_val = int(0.15 * n_total)
    n_test = n_total - n_train - n_val

    generador = torch.Generator().manual_seed(SEED)
    train_ds, val_ds, test_ds = random_split(
        dataset, [n_train, n_val, n_test], generator=generador
    )

    test_data_imgs = [Path(dataset.imgs[x][0]) for x in test_ds.indices]

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size)
    test_dl = DataLoader(test_ds, batch_size=batch_size)
    return train_dl, val_dl, test_dl, dataset.classes, test_data_imgs


train_dl, val_dl, test_dl, CLASES, test_paths = crear_dataloaders(transform)

print(f"Clases (orden alfabético): {CLASES}")
print(f"Train: {len(train_dl.dataset)} | Val: {len(val_dl.dataset)} | Test: {len(test_dl.dataset)}")

## 3. El modelo: ResNet-18 entrenada

Grad-CAM **no entrena nada**: solo *explica* un modelo que ya existe. Así que primero necesitamos un modelo entrenado.

Tenemos dos caminos:

1. **Cargar** los pesos que guardamos en el workshop (`../models/resnet18_rps.pt`) → instantáneo.
2. Si ese archivo no existe (o pones `ENTRENAR_DE_NUEVO = True`), **entrenamos** con fine-tuning unas pocas épocas → unos minutos.

Para el segundo camino reusamos las funciones de entrenamiento del workshop: `entrenar_una_epoca()`, `evaluar()` y `fit()`.

In [ ]:
def entrenar_una_epoca(modelo, dataloader, criterio, optimizador):
    # Una pasada completa por los datos de entrenamiento.
    modelo.train()
    perdida_total, aciertos = 0.0, 0

    for imagenes, etiquetas in dataloader:
        imagenes = imagenes.to(DEVICE)
        etiquetas = etiquetas.to(DEVICE)

        optimizador.zero_grad()                 # 0. limpiar gradientes anteriores
        logits = modelo(imagenes)               # 1. forward: la red predice
        perdida = criterio(logits, etiquetas)   # 2. loss: ¿qué tan mal?
        perdida.backward()                      # 3. backward: calcular gradientes
        optimizador.step()                      # 4. step: ajustar pesos

        perdida_total += perdida.item() * len(etiquetas)
        aciertos += (logits.argmax(dim=1) == etiquetas).sum().item()

    n = len(dataloader.dataset)
    return perdida_total / n, aciertos / n


@torch.no_grad()
def evaluar(modelo, dataloader, criterio):
    # Mide loss y accuracy sin modificar el modelo.
    modelo.eval()
    perdida_total, aciertos = 0.0, 0

    for imagenes, etiquetas in dataloader:
        imagenes = imagenes.to(DEVICE)
        etiquetas = etiquetas.to(DEVICE)
        logits = modelo(imagenes)
        perdida_total += criterio(logits, etiquetas).item() * len(etiquetas)
        aciertos += (logits.argmax(dim=1) == etiquetas).sum().item()

    n = len(dataloader.dataset)
    return perdida_total / n, aciertos / n


def fit(modelo, train_dl, val_dl, criterio, optimizador, epocas):
    # Entrena el modelo y muestra las métricas de cada época.
    for epoca in range(1, epocas + 1):
        train_loss, train_acc = entrenar_una_epoca(modelo, train_dl, criterio, optimizador)
        val_loss, val_acc = evaluar(modelo, val_dl, criterio)
        print(f"Época {epoca}/{epocas} | "
              f"train loss: {train_loss:.4f}, acc: {train_acc:.2%} | "
              f"val loss: {val_loss:.4f}, acc: {val_acc:.2%}")

In [ ]:
RUTA_PESOS = Path("../models/resnet18_rps.pt")
ENTRENAR_DE_NUEVO = False     # ponlo en True para entrenar aunque exista el archivo

criterio = nn.CrossEntropyLoss()

if RUTA_PESOS.exists() and not ENTRENAR_DE_NUEVO:
    # Camino 1: misma arquitectura + pesos guardados en el workshop
    modelo = timm.create_model("resnet18", pretrained=False, num_classes=3)
    modelo.load_state_dict(torch.load(RUTA_PESOS, map_location="cpu"))
    modelo = modelo.to(DEVICE)
    print(f"Pesos cargados desde {RUTA_PESOS}")
else:
    # Camino 2: fine-tuning completo de ResNet-18 pre-entrenada en ImageNet
    modelo = timm.create_model("resnet18", pretrained=True, num_classes=3).to(DEVICE)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-4)
    fit(modelo, train_dl, val_dl, criterio, optimizador, epocas=5)

test_loss, test_acc = evaluar(modelo, test_dl, criterio)
print(f"Accuracy en test: {test_acc:.2%}")

¡El modelo es muy bueno! Pero **un accuracy alto no garantiza que el modelo razone bien**. Hay casos famosos de modelos que "acertaban" por las razones equivocadas:

- 🐺 Un clasificador de *lobo vs. husky* que en realidad detectaba **nieve** en el fondo (los lobos solían fotografiarse en la nieve).
- 🏥 Modelos de radiografías que aprendían a leer **la etiqueta del hospital** impresa en la placa, en lugar de mirar el pulmón.

Por eso vale la pena preguntarle al modelo: *"¿en qué te fijaste?"*.

## 4. Antes de Grad-CAM: ¿qué "ve" una CNN por dentro?

Recuerda cómo funciona una CNN: cada capa convolucional aplica muchos **filtros**, y cada filtro produce un **feature map** (mapa de características) que indica *dónde* encontró su patrón.

```
imagen 3×224×224 → [capas conv...] → 512 feature maps de 7×7 → pooling → 3 salidas (logits)
                                      ▲
                                      └── aquí mira Grad-CAM
```

En la **última capa convolucional** de ResNet-18 (`layer4`) hay **512 feature maps de 7×7**:

- Son la **última etapa donde todavía hay información espacial** (una grilla de 7×7 que "cubre" la imagen). Después, el *pooling* la aplasta a 512 números y se pierde el *dónde*.
- Son las **características más abstractas** que aprendió la red: ya no bordes simples, sino cosas como "dedos extendidos" o "puño cerrado".

`timm` nos facilita obtenerlos, porque divide el modelo en dos mitades:

| Método | ¿Qué hace? | Salida |
|--------|-----------|--------|
| `modelo.forward_features(x)` | Todas las capas convolucionales | `[1, 512, 7, 7]` |
| `modelo.forward_head(A)` | Pooling + capa lineal final | `[1, 3]` (logits) |

In [ ]:
# Tomamos una imagen del test (el modelo nunca la vio durante el entrenamiento)
ruta_ejemplo = test_paths[0]
imagen_pil = Image.open(ruta_ejemplo).convert("RGB")

x = transform(imagen_pil).unsqueeze(0).to(DEVICE)   # añadimos la dimensión de batch → [1, 3, 224, 224]
print(f"Imagen: {ruta_ejemplo.name} | clase real: {ruta_ejemplo.parent.name}")
print(f"Tensor de entrada: {tuple(x.shape)}")

plt.figure(figsize=(4, 4))
plt.imshow(desnormalizar(x[0]))
plt.title(f"Clase real: {ruta_ejemplo.parent.name}")
plt.axis("off")
plt.show()

Pasamos la imagen por la **primera mitad** del modelo y miramos los feature maps.

> ⚠️ `modelo.eval()` es importante: pone al modelo en **modo evaluación** (igual que al predecir). No queremos explicar un modelo que se comporta distinto al que usamos de verdad.

In [ ]:
modelo.eval()

activaciones = modelo.forward_features(x)     # A: los feature maps de la última capa conv
print(f"Feature maps (activaciones): {tuple(activaciones.shape)}  ← [batch, canales, alto, ancho]")

# Mostramos los 8 feature maps que más se "encendieron" (mayor activación promedio)
promedio_por_canal = activaciones[0].mean(dim=(1, 2))
top_canales = promedio_por_canal.argsort(descending=True)[:8]

fig, axes = plt.subplots(1, 9, figsize=(18, 2.6))
axes[0].imshow(desnormalizar(x[0]))
axes[0].set_title("Imagen")
for ax, canal in zip(axes[1:], top_canales):
    ax.imshow(activaciones[0, canal].detach().cpu(), cmap="viridis")
    ax.set_title(f"Canal {canal.item()}")
for ax in axes:
    ax.axis("off")
plt.suptitle("Algunos de los 512 feature maps de 7×7 (amarillo = el filtro detectó su patrón ahí)", y=1.03)
plt.tight_layout()
plt.show()

### ¿Qué observamos?

- Cada feature map es una grilla de **7×7**: cada casilla corresponde, aproximadamente, a un bloque de 32×32 píxeles de la imagen original (224 / 7 = 32).
- Cada canal se "enciende" en zonas distintas: cada uno detecta **un patrón diferente**.

El problema: hay **512** mapas. ¿Cuáles importan para decidir "piedra", "papel" o "tijeras"? No todos pesan igual en la decisión... y aquí entran los **gradientes**.

## 5. Grad-CAM paso a paso

La idea en una frase:

> **Grad-CAM = promedio ponderado de los feature maps, donde el peso de cada mapa es "cuánto importa para la clase que queremos explicar".**

¿Y cómo medimos "cuánto importa"? Con el **gradiente**: la misma herramienta que usamos para entrenar (`loss.backward()`), pero ahora con otro propósito.

> 📐 **Recordatorio:** el gradiente $\frac{\partial y}{\partial A}$ responde a *"si aumento un poquito $A$, ¿cuánto cambia $y$?"*. Si el gradiente es grande y positivo, ese feature map **empuja fuerte** hacia la clase.

Lo haremos en 5 pasos:

| Paso | Operación | Forma del resultado |
|------|-----------|---------------------|
| 1 | Forward: obtener feature maps $A$ y logits | $A$: `[1, 512, 7, 7]` |
| 2 | Backward desde el puntaje de la clase $y^c$ | gradientes: `[1, 512, 7, 7]` |
| 3 | Promediar gradientes → un **peso** por canal $\alpha_k$ | `[512]` |
| 4 | Suma ponderada + ReLU | mapa `[7, 7]` |
| 5 | Agrandar a 224×224 y superponer | mapa `[224, 224]` |

### Paso 1 — Forward: feature maps y predicción

Hacemos el forward **en dos mitades**, para quedarnos con las activaciones intermedias $A$.

`activaciones.retain_grad()` le dice a PyTorch: *"cuando hagas backward, **guárdame** el gradiente de este tensor"*. Normalmente PyTorch solo guarda gradientes de los pesos (para entrenar) y descarta los intermedios para ahorrar memoria.

In [ ]:
modelo.eval()

activaciones = modelo.forward_features(x)     # 1ª mitad: capas convolucionales → A
activaciones.retain_grad()                    # queremos el gradiente de A después del backward
logits = modelo.forward_head(activaciones)    # 2ª mitad: pooling + capa lineal → 3 logits

probabilidades = torch.softmax(logits, dim=1)[0]
clase_predicha = logits.argmax(dim=1).item()

print(f"Logits: {logits[0].detach().cpu().numpy().round(2)}")
for clase, prob in zip(CLASES, probabilidades):
    print(f"  {clase:>10s}: {prob:.1%}")
print(f"\nPredicción: {CLASES[clase_predicha]}  (real: {ruta_ejemplo.parent.name})")

### Paso 2 — Backward: ¿cómo afecta cada feature map al puntaje de la clase?

Elegimos la clase que queremos explicar (la predicha) y hacemos backward desde **su logit** $y^c$.

Fíjate en la diferencia con el entrenamiento:

| | Entrenamiento | Grad-CAM |
|---|---|---|
| Backward desde... | la **loss** | el **puntaje de una clase** $y^c$ |
| Gradientes que nos interesan | los de los **pesos** | los de las **activaciones** $A$ |
| ¿Actualizamos pesos? | Sí (`optimizador.step()`) | **No**, el modelo no cambia |

> 💡 Usamos el **logit** (antes del softmax) y no la probabilidad: el softmax "se satura" cerca de 100% y los gradientes se vuelven casi cero.

In [ ]:
modelo.zero_grad()                          # limpiamos gradientes previos, por si acaso
puntaje_clase = logits[0, clase_predicha]   # y^c: el logit de la clase a explicar
puntaje_clase.backward()                    # calcula ∂y^c/∂A para cada casilla de cada feature map

gradientes = activaciones.grad
print(f"Gradientes: {tuple(gradientes.shape)}  ← misma forma que las activaciones")

### Paso 3 — Un peso por feature map ($\alpha_k$)

Tenemos un gradiente por **cada casilla** (7×7) de **cada** feature map (512). Para resumir, **promediamos** los gradientes de cada mapa → un solo número por canal:

$$\alpha_k^c = \frac{1}{7 \times 7} \sum_{i}\sum_{j} \frac{\partial y^c}{\partial A^k_{ij}}$$

- $\alpha_k$ **alto y positivo** → el canal $k$ es evidencia **a favor** de la clase $c$.
- $\alpha_k$ **negativo** → el canal $k$ es evidencia **en contra** de la clase.

In [ ]:
pesos = gradientes[0].mean(dim=(1, 2))      # promedio sobre alto y ancho → [512]
print(f"Pesos α: {tuple(pesos.shape)}")

pesos_np = pesos.detach().cpu().numpy()
orden = np.sort(pesos_np)[::-1]

fig, ax = plt.subplots(figsize=(12, 3.5))
colores = ["#c0392b" if p > 0 else "#2980b9" for p in orden]
ax.bar(range(len(orden)), orden, color=colores, width=1.0)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Canales (ordenados de mayor a menor peso)")
ax.set_ylabel("Peso α")
ax.set_title(f"Importancia de los 512 canales para la clase '{CLASES[clase_predicha]}' "
             "(rojo = a favor, azul = en contra)")
plt.show()

### ¿Qué observamos?

- Aproximadamente **la mitad** de los canales suma a favor de la clase y la otra mitad resta.
- **Ningún canal decide solo**: los pesos bajan de forma gradual. La decisión es la **suma de muchas pequeñas pistas**, no un único "detector de tijeras".

Veamos los feature maps de los canales **más importantes** (los de mayor $\alpha$). Compáralos con los de la sección 4 (los que más se "encendían"): **encenderse mucho no es lo mismo que importar para la decisión**.

In [ ]:
top_por_peso = pesos.argsort(descending=True)[:8]

fig, axes = plt.subplots(1, 9, figsize=(18, 2.6))
axes[0].imshow(desnormalizar(x[0]))
axes[0].set_title("Imagen")
for ax, canal in zip(axes[1:], top_por_peso):
    ax.imshow(activaciones[0, canal].detach().cpu(), cmap="viridis")
    ax.set_title(f"Canal {canal.item()}\nα = {pesos[canal]:.4f}", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.suptitle(f"Los 8 canales más importantes para decir '{CLASES[clase_predicha]}'", y=1.05)
plt.tight_layout()
plt.show()

> 💡 Si algún mapa sale **completamente oscuro**, ese canal es importante *en general* para la clase, pero **no se activó en esta imagen** (su activación es 0). Grad-CAM necesita las dos cosas: que el canal **importe** ($\alpha$ alto) **y** que esté **encendido** ($A$ alto). Eso es justo lo que hace la multiplicación del siguiente paso.

### Paso 4 — Suma ponderada + ReLU → el mapa de Grad-CAM

Combinamos los 512 feature maps, cada uno multiplicado por su peso:

$$L^c_{\text{Grad-CAM}} = \text{ReLU}\left(\sum_k \alpha_k^c \, A^k\right)$$

- La **suma ponderada** junta toda la evidencia en un solo mapa de 7×7.
- La **ReLU** (negativos → 0) se queda solo con las zonas que **suman a favor** de la clase. Las que restan no nos interesan para responder *"¿por qué dijiste esto?"*.

In [ ]:
# pesos: [512] → [512, 1, 1] para multiplicar cada feature map [512, 7, 7] por su peso
mapa = (pesos.view(-1, 1, 1) * activaciones[0]).sum(dim=0)   # suma ponderada → [7, 7]
mapa = F.relu(mapa)                                           # solo evidencia positiva

mapa_np = mapa.detach().cpu().numpy()
print(f"Mapa Grad-CAM: {mapa_np.shape}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
ax1.imshow(desnormalizar(x[0]))
ax1.set_title("Imagen")
ax1.axis("off")

im = ax2.imshow(mapa_np, cmap="jet")
for i in range(7):
    for j in range(7):
        ax2.text(j, i, f"{mapa_np[i, j]:.2f}", ha="center", va="center", color="white", fontsize=7)
ax2.set_title("Mapa Grad-CAM 7×7 (valores reales)")
ax2.axis("off")
plt.colorbar(im, ax=ax2, fraction=0.046)
plt.tight_layout()
plt.show()

Ya tenemos la respuesta... ¡pero en una grilla de 7×7! Cada casilla corresponde a un bloque de la imagen. Las casillas "calientes" son las zonas que más aportaron a la decisión.

### Paso 5 — Agrandar y superponer

Para verlo sobre la foto:

1. **Normalizamos** el mapa entre 0 y 1 (solo nos importa la importancia *relativa* dentro de la imagen).
2. Lo **agrandamos** de 7×7 a 224×224 con interpolación bilineal (`F.interpolate`), que suaviza los bordes entre casillas.
3. Lo **superponemos** a la imagen con transparencia (`alpha`).

In [ ]:
# 1. Normalizar a [0, 1]
mapa_norm = (mapa - mapa.min()) / (mapa.max() - mapa.min() + 1e-8)

# 2. Agrandar: F.interpolate espera [batch, canales, alto, ancho] → añadimos 2 dimensiones
mapa_grande = F.interpolate(
    mapa_norm[None, None], size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False
)[0, 0]
mapa_grande = mapa_grande.detach().cpu().numpy()
print(f"Mapa agrandado: {mapa_grande.shape}")

# 3. Superponer
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(desnormalizar(x[0]))
axes[0].set_title(f"Imagen (real: {ruta_ejemplo.parent.name})")

axes[1].imshow(mapa_grande, cmap="jet")
axes[1].set_title("Mapa de calor Grad-CAM")

axes[2].imshow(desnormalizar(x[0]))
axes[2].imshow(mapa_grande, cmap="jet", alpha=0.45)
axes[2].set_title(f"Predicción: {CLASES[clase_predicha]} ({probabilidades[clase_predicha]:.1%})")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

**¡Ese es Grad-CAM!** Las zonas rojas son las que más pesaron para que la red dijera su predicción.

Fíjate en qué partes de la mano se ilumina: ¿los dedos? ¿el puño? ¿el borde de la mano contra el fondo? Eso es lo que la red considera **evidencia** de la clase.

Resumen de lo que hicimos, en pseudo-código:

```
A          = feature maps de la última capa conv          # [512, 7, 7]
gradientes = ∂(logit de la clase) / ∂A                    # [512, 7, 7]
α          = promedio de gradientes de cada canal         # [512]
mapa       = ReLU(Σ α_k · A_k)                            # [7, 7]
mapa       = agrandar(normalizar(mapa))                   # [224, 224]
```

## 6. Empaquetamos todo en funciones reutilizables

Juntamos los 5 pasos en `calcular_gradcam()`. Tiene un parámetro clave: `clase`.

- `clase=None` → explica la clase **predicha** ("¿por qué dijiste lo que dijiste?").
- `clase=1` → explica **cualquier clase** que elijamos ("¿qué te haría pensar que es *rock*?").

Y `mostrar_gradcam()` se encarga de dibujar imagen original + mapa + superposición a partir de la ruta de una imagen.

In [ ]:
def calcular_gradcam(modelo, x, clase=None):
    # Devuelve (mapa Grad-CAM de 224×224 en [0, 1], clase explicada, probabilidades).
    modelo.eval()

    # Paso 1: forward en dos mitades, guardando el gradiente de las activaciones
    activaciones = modelo.forward_features(x)
    activaciones.retain_grad()
    logits = modelo.forward_head(activaciones)
    if clase is None:
        clase = logits.argmax(dim=1).item()

    # Paso 2: backward desde el logit de la clase
    modelo.zero_grad()
    logits[0, clase].backward()

    # Paso 3: un peso por canal = promedio de sus gradientes
    pesos = activaciones.grad[0].mean(dim=(1, 2))

    # Paso 4: suma ponderada de los feature maps + ReLU
    mapa = F.relu((pesos.view(-1, 1, 1) * activaciones[0]).sum(dim=0))

    # Paso 5: normalizar a [0, 1] y agrandar al tamaño de la imagen
    mapa = (mapa - mapa.min()) / (mapa.max() - mapa.min() + 1e-8)
    mapa = F.interpolate(mapa[None, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]

    probabilidades = torch.softmax(logits, dim=1)[0].detach().cpu()
    return mapa.detach().cpu().numpy(), clase, probabilidades


def cargar_imagen(ruta):
    # Lee una imagen del disco y la convierte al tensor que espera el modelo: [1, 3, 224, 224].
    imagen = Image.open(ruta).convert("RGB")
    return transform(imagen).unsqueeze(0).to(DEVICE)


def mostrar_gradcam(ruta, modelo=modelo, clase=None):
    # Muestra imagen original, mapa de calor y superposición.
    x = cargar_imagen(ruta)
    mapa, clase, probabilidades = calcular_gradcam(modelo, x, clase)
    imagen = desnormalizar(x[0])

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(imagen)
    axes[0].set_title(f"Real: {Path(ruta).parent.name}")
    axes[1].imshow(mapa, cmap="jet")
    axes[1].set_title("Grad-CAM")
    axes[2].imshow(imagen)
    axes[2].imshow(mapa, cmap="jet", alpha=0.45)
    axes[2].set_title(f"Explicando '{CLASES[clase]}' ({probabilidades[clase]:.1%})")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# Comprobamos que la función da lo mismo que el paso a paso
mostrar_gradcam(ruta_ejemplo)

## 7. Galería: ¿en qué se fija la red para cada clase?

Un solo ejemplo puede engañar. Miremos **varias imágenes de test de cada clase** y busquemos **patrones**:

- ¿La red mira la **mano** o el **fondo**?
- ¿Mira **toda** la mano o solo **una parte** (los dedos, los nudillos)?
- ¿Se fija en lo mismo para las 3 clases?

In [ ]:
N_POR_CLASE = 5

fig, axes = plt.subplots(len(CLASES), N_POR_CLASE, figsize=(3 * N_POR_CLASE, 3.2 * len(CLASES)))

for fila, nombre_clase in enumerate(CLASES):
    rutas_clase = [r for r in test_paths if r.parent.name == nombre_clase][:N_POR_CLASE]
    for col, ruta in enumerate(rutas_clase):
        x_img = cargar_imagen(ruta)
        mapa, pred, probs = calcular_gradcam(modelo, x_img)

        ax = axes[fila, col]
        ax.imshow(desnormalizar(x_img[0]))
        ax.imshow(mapa, cmap="jet", alpha=0.45)
        color = "green" if CLASES[pred] == nombre_clase else "red"
        ax.set_title(f"pred: {CLASES[pred]} ({probs[pred]:.0%})", color=color, fontsize=10)
        ax.axis("off")
    axes[fila, 0].text(-0.1, 0.5, nombre_clase, transform=axes[fila, 0].transAxes,
                       rotation=90, va="center", ha="right", fontsize=13, fontweight="bold")

plt.suptitle("Grad-CAM de la clase predicha (título verde = acierto, rojo = error)", fontsize=14)
plt.tight_layout()
plt.show()

### ¿Qué patrones ves?

Tómate un minuto para observar la galería antes de leer. Algunas preguntas guía:

- **Tijeras:** ¿se ilumina la zona de los **dos dedos** extendidos en "V"?
- **Papel:** ¿la red mira los **dedos abiertos** / la palma completa?
- **Piedra:** ¿se fija en el **puño** / los nudillos?
- ¿Hay mapas de calor sobre el **fondo verde** o en los **bordes** de la imagen? Si los hubiera, sería una señal de alerta: el modelo estaría usando pistas que no tienen que ver con la mano.

> Si la red se fija en las partes que **un humano** usaría para decidir, podemos confiar un poco más en ella. Si se fija en cosas raras, tenemos un problema... aunque el accuracy sea altísimo.

**💬 Lo que vemos con el modelo del workshop:**

- **Papel:** el calor cae sobre la **base de los dedos extendidos** y la palma.
- **Piedra:** el calor cae sobre el **puño / los nudillos**.
- **Tijeras:** el calor cae justo en la **unión de los dos dedos en "V"**.
- El **fondo verde** casi no se ilumina y el **brazo** tampoco: la red mira la parte de la mano que define el gesto. ¡Buena señal!

## 8. Una imagen, tres explicaciones: "¿por qué X y no Y?"

Grad-CAM es **específico por clase**: el mapa cambia según la clase que le pidamos explicar. Podemos preguntarle a la red, sobre la **misma** imagen:

- *"¿Qué parte te hace pensar que es **papel**?"*
- *"¿Qué parte te hace pensar que es **piedra**?"*
- *"¿Qué parte te hace pensar que es **tijeras**?"*

Esto es posible gracias al **Paso 2**: hacemos backward desde el logit de la clase que elijamos, no necesariamente la predicha.

In [ ]:
def comparar_clases(ruta, modelo=modelo):
    # Grad-CAM de la misma imagen para cada una de las clases.
    x_img = cargar_imagen(ruta)
    imagen = desnormalizar(x_img[0])

    fig, axes = plt.subplots(1, len(CLASES) + 1, figsize=(4 * (len(CLASES) + 1), 4))
    axes[0].imshow(imagen)
    axes[0].set_title(f"Real: {Path(ruta).parent.name}", fontweight="bold")

    for ax, indice_clase in zip(axes[1:], range(len(CLASES))):
        mapa, _, probs = calcular_gradcam(modelo, x_img, clase=indice_clase)
        ax.imshow(imagen)
        ax.imshow(mapa, cmap="jet", alpha=0.45)
        ax.set_title(f"¿Por qué '{CLASES[indice_clase]}'? ({probs[indice_clase]:.1%})")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# Una imagen de tijeras del test
ruta_tijeras = next(r for r in test_paths if r.parent.name == "scissors")
comparar_clases(ruta_tijeras)

In [ ]:
# Probemos con una imagen de piedra
ruta_piedra = next(r for r in test_paths if r.parent.name == "rock")
comparar_clases(ruta_piedra)

### ¿Cómo leer estos mapas?

- En la **clase predicha** (probabilidad alta), el mapa suele concentrarse en la parte de la mano que "define" el gesto.
- En las **otras clases**, la red busca "la poca evidencia que habría a favor" de esa clase: a veces cae en zonas distintas, y a veces el mapa sale casi vacío o disperso, porque **no hay casi nada a favor**.

Fíjate que, en las clases con probabilidad muy baja, el mapa a veces cae **fuera de la mano**: en el fondo o en las pequeñas marcas del borde izquierdo de las fotos. No es que la red "vea piedra" en el fondo: es que **no encuentra evidencia en la mano** y lo poco que queda está disperso.

**Ojo con la normalización:** cada mapa se normaliza entre 0 y 1 **por separado**. Un mapa muy rojo para una clase con 0.1% de probabilidad **no** significa que haya mucha evidencia: solo muestra dónde está la *poca* evidencia que hay. Mira siempre el mapa **junto** con la probabilidad.

## 9. Depurando errores: ¿por qué se equivocó?

Grad-CAM es especialmente útil para entender los **errores** del modelo. Buscamos en todo el test:

1. Las imágenes que el modelo **clasificó mal**.
2. Si no hay errores (¡puede pasar con un modelo tan bueno!), las imágenes donde estuvo **menos seguro** (menor confianza en su predicción).

In [ ]:
@torch.no_grad()
def predecir_todo(modelo, rutas):
    # Devuelve una lista de (ruta, clase real, clase predicha, confianza) para cada imagen.
    modelo.eval()
    resultados = []
    for ruta in rutas:
        probs = torch.softmax(modelo(cargar_imagen(ruta)), dim=1)[0]
        pred = probs.argmax().item()
        resultados.append((ruta, CLASES.index(ruta.parent.name), pred, probs[pred].item()))
    return resultados


resultados_test = predecir_todo(modelo, test_paths)
errores = [r for r in resultados_test if r[1] != r[2]]
print(f"Errores en test: {len(errores)} de {len(resultados_test)}")

for ruta, real, pred, conf in errores:
    print(f"  {ruta.name}: real = {CLASES[real]:>8s} | predicho = {CLASES[pred]:>8s} ({conf:.1%})")

In [ ]:
# Si hay errores los mostramos; si no, mostramos los casos de menor confianza
if errores:
    casos = errores[:4]
    titulo = "❌ Errores del modelo"
else:
    casos = sorted(resultados_test, key=lambda r: r[3])[:4]
    titulo = "🤔 Casos donde el modelo estuvo menos seguro"

print(titulo)
for ruta, real, pred, conf in casos:
    comparar_clases(ruta)

### ¿Qué nos dicen los errores?

** En nuestro caso** (modelo del workshop), los errores son fotos de **papel** clasificadas como **tijeras**. Mira la columna *"¿Por qué 'scissors'?"*: el mapa se concentra en **dedos que están juntos de a pares**, formando una especie de "V". ¡Desde el punto de vista de la red, esa mano *parece* tijeras! El error tiene una explicación razonable.

En general, al mirar errores (o dudas) con Grad-CAM, suelen aparecer explicaciones como:

- La mano está **parcialmente fuera** de la imagen o en una **posición inusual**.
- El gesto es **ambiguo**: dedos a medio abrir (¿piedra o papel?), o un dedo doblado (¿tijeras o papel?).
- La red se fijó en una **zona equivocada** (una sombra, el brazo, el borde).

Esto es oro para mejorar un modelo: nos dice **qué tipo de datos faltan** (p. ej. más fotos con la mano rotada) o si hace falta *data augmentation*.

## 10. Prueba de cordura: ¿los mapas dependen de lo que aprendió la red?

Una buena pregunta científica: *"¿Y si Grad-CAM siempre marca la mano, **sin importar** lo que haya aprendido la red?"*. Si fuera así, el método no explicaría nada.

**Experimento:** comparamos los mapas de nuestro modelo entrenado contra los de una ResNet-18 con **pesos aleatorios** (que nunca aprendió nada).

- Si los mapas del modelo aleatorio se ven **igual** de razonables → Grad-CAM no es confiable.
- Si se ven **ruidosos / sin sentido** → los mapas del modelo entrenado sí reflejan **lo que aprendió**.

> Esta idea viene del paper *Sanity Checks for Saliency Maps* (Adebayo et al., 2018), que mostró que algunos métodos de explicación **fallan** esta prueba. Grad-CAM la pasa.

In [ ]:
modelo_aleatorio = timm.create_model("resnet18", pretrained=False, num_classes=3).to(DEVICE)

rutas_cordura = [next(r for r in test_paths if r.parent.name == c) for c in CLASES]

fig, axes = plt.subplots(2, len(rutas_cordura), figsize=(4 * len(rutas_cordura), 8))
for col, ruta in enumerate(rutas_cordura):
    x_img = cargar_imagen(ruta)
    imagen = desnormalizar(x_img[0])
    for fila, (nombre, m) in enumerate([("Entrenado", modelo), ("Aleatorio", modelo_aleatorio)]):
        mapa, pred, probs = calcular_gradcam(m, x_img)
        axes[fila, col].imshow(imagen)
        axes[fila, col].imshow(mapa, cmap="jet", alpha=0.45)
        axes[fila, col].set_title(f"{nombre} → {CLASES[pred]} ({probs[pred]:.0%})")
        axes[fila, col].axis("off")

plt.suptitle("Modelo entrenado (arriba) vs. modelo con pesos aleatorios (abajo)", fontsize=14)
plt.tight_layout()
plt.show()

### ¿Qué observamos?

| | Modelo entrenado | Modelo aleatorio |
|---|---|---|
| Predicción | Correcta y con confianza alta | Siempre ~33-35%: adivina |
| Mapa | **Compacto**, en la zona que define el gesto | **Difuso**, se derrama sobre el fondo |
| ¿Cambia según el gesto? | Sí: puño, "V", dedos abiertos | No: marca la mano de forma genérica |

🤔 Curiosamente, incluso la red aleatoria "encuentra" la mano: es el único objeto con bordes y textura en una foto de fondo liso, y hasta filtros aleatorios reaccionan a eso. Por eso la prueba de cordura es importante: **un mapa sobre la mano no basta** para decir que el modelo razona bien; hay que comparar con una referencia.

**Conclusión:** los mapas del modelo entrenado son claramente más específicos que los del aleatorio → Grad-CAM **sí refleja** lo que aprendió la red. ✅

## 11. Bonus: ¿y si miramos capas más tempranas?

Hasta ahora usamos la **última** capa convolucional (`layer4`, 7×7). ¿Por qué esa y no otra? Comparemos con capas anteriores de ResNet-18:

| Capa | Tamaño del feature map | ¿Qué detecta? |
|------|-----------------------|---------------|
| `layer2` | 28×28 | Patrones simples: bordes, texturas → **mucho detalle, poco significado** |
| `layer3` | 14×14 | Partes: curvas, formas de dedos |
| `layer4` | 7×7 | Conceptos: "puño", "dedos en V" → **poco detalle, mucho significado** |

Para capas intermedias ya no podemos usar `forward_features` (que siempre llega hasta el final). Usamos un **hook** ("gancho"): una función que PyTorch ejecuta automáticamente cuando los datos pasan por una capa, y que nos permite "espiar" y guardar su salida.

In [ ]:
def calcular_gradcam_capa(modelo, x, capa, clase=None):
    # Grad-CAM en cualquier capa convolucional, usando un forward hook.
    modelo.eval()
    guardado = {}

    def espiar_salida(modulo, entrada, salida):
        salida.retain_grad()                 # guardar su gradiente en el backward
        guardado["activaciones"] = salida    # guardar las activaciones

    gancho = capa.register_forward_hook(espiar_salida)   # colgamos el gancho en la capa
    logits = modelo(x)                                   # forward normal: el gancho se ejecuta solo
    gancho.remove()                                      # lo quitamos para no dejarlo colgado

    if clase is None:
        clase = logits.argmax(dim=1).item()
    modelo.zero_grad()
    logits[0, clase].backward()

    # A partir de aquí, exactamente los pasos 3, 4 y 5 de antes
    activaciones = guardado["activaciones"]
    pesos = activaciones.grad[0].mean(dim=(1, 2))
    mapa = F.relu((pesos.view(-1, 1, 1) * activaciones[0]).sum(dim=0))
    tamano_original = tuple(mapa.shape)
    mapa = (mapa - mapa.min()) / (mapa.max() - mapa.min() + 1e-8)
    mapa = F.interpolate(mapa[None, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    return mapa.detach().cpu().numpy(), tamano_original


capas = {"layer2": modelo.layer2, "layer3": modelo.layer3, "layer4": modelo.layer4}

for ruta in [ruta_tijeras, ruta_piedra]:
    x_img = cargar_imagen(ruta)
    imagen = desnormalizar(x_img[0])

    fig, axes = plt.subplots(1, len(capas) + 1, figsize=(16, 4))
    axes[0].imshow(imagen)
    axes[0].set_title(f"Real: {ruta.parent.name}", fontweight="bold")
    for ax, (nombre_capa, capa) in zip(axes[1:], capas.items()):
        mapa, tamano = calcular_gradcam_capa(modelo, x_img, capa)
        ax.imshow(imagen)
        ax.imshow(mapa, cmap="jet", alpha=0.45)
        ax.set_title(f"{nombre_capa} ({tamano[0]}×{tamano[1]})")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

### ¿Qué observamos?

- **Capas tempranas** (`layer2`): mapas con más detalle, pero **ruidosos** — marcan bordes y texturas que no son específicos de la clase.
- **Última capa** (`layer4`): mapas **borrosos** (solo 7×7), pero **enfocados** en lo que define la clase.

Por eso el estándar es usar la **última capa convolucional**: es el mejor equilibrio entre *dónde* (información espacial) y *qué* (significado de alto nivel).

## 12. Cierre: ¿qué aprendimos?

**Grad-CAM en 4 ideas:**

1. **Qué es:** un mapa de calor que muestra **qué zonas de la imagen** usó una CNN para decidir una clase.
2. **Cómo funciona:** toma los **feature maps** de la última capa convolucional y los combina, pesando cada uno por el **promedio de su gradiente** respecto al puntaje de la clase.
3. **Es específico por clase:** podemos preguntar *"¿por qué papel?"* y *"¿por qué piedra?"* sobre la misma imagen.
4. **No reemplaza al accuracy, lo complementa:** el accuracy dice *cuánto* acierta el modelo; Grad-CAM ayuda a entender *por qué*.

**Limitaciones (para no sobreinterpretar):**

| Limitación | Consecuencia |
|-----------|--------------|
| Resolución baja (7×7 en ResNet-18) | Dice *qué zona*, no *qué píxel exacto* |
| Solo muestra evidencia **positiva** (ReLU) | No muestra lo que va *en contra* de la clase |
| Normalizado por imagen | Compara zonas **dentro** de una imagen, no entre imágenes |
| Necesita capas convolucionales | No sirve tal cual para un MLP (¡no tiene feature maps espaciales!) |